# <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;  " > 实践实验室：基于内容过滤的深度学习

在本练习中，你将使用神经网络实现基于内容的过滤，以构建一个电影推荐系统。

# 大纲 <img align="left" src="./images/film_reel.png"     style=" width:40px;  " >
- [1 - 导入包](#1)
- [2 - 电影评分数据集](#2)
  - [2.1 基于神经网络的内容过滤](#2.1)
  - [2.2 准备训练数据](#2.2)
- [3 - 基于内容过滤的神经网络](#3)
  - [3.1 预测](#3.1)
    - [练习1](#ex01)
- [4 - 恭喜！](#4)


<a name="1"></a>
## 1 - 包 <img align="left" src="./images/movie_camera.png"     style=" width:40px;  ">
我们将使用熟悉的软件包，如NumPy、TensorFlow，以及来自[scikit-learn](https://scikit-learn.org/stable/)的实用程序。我们还将使用[tabulate](https://pypi.org/project/tabulate/)来整齐地打印表格，并使用[Pandas](https://pandas.pydata.org/)来整理表格数据。

In [1]:
import numpy as np
import numpy.ma as ma
from numpy import genfromtxt
from collections import defaultdict
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
from recsysNN_utils import *
pd.set_option("display.precision", 1)

<a name="2"></a>
## 2 - 电影评分数据集 <img align="left" src="./images/film_rating.png" style=" width:40px;" >
该数据集源自[MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/)数据集。

[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集包含9000部电影，由600名用户进行评分，评分范围为0.5到5，步长为0.5。为聚焦于2000年以来的电影和热门类型，数据集规模已缩小。缩小后的数据集有$n_u = 395$名用户和$n_m = 694$部电影。   

对于每部电影，数据集提供电影标题、发行日期以及一种或多种类型。例如，《玩具总动员3》于2010年发行，有多种类型：“冒险|动画|儿童|喜剧|奇幻|IMAX”。   

该数据集除了用户评分外，几乎不包含关于用户的其他信息。此数据集用于为下文所述的神经网络创建训练向量。 

<a name="2.1"></a>
### 2.1 基于神经网络的基于内容的过滤
在协同过滤实验中，你生成了两个向量，一个用户向量和一个物品/电影向量，它们的点积可以预测评分。这些向量完全由评分推导出。   

基于内容的过滤同样会生成一个用户和电影特征向量，但它认识到可能存在关于用户和电影的其他信息，这些信息可能会改进预测。这些额外信息会被提供给一个神经网络，然后该神经网络会生成如下所示的用户和电影向量。   

<figure>
    <center> <img src="./images/RecSysNN.png"   style="width:500px;height:280px;" ></center>
</figure>   

提供给网络的电影内容是原始数据和一些“**工程特征**”的组合(回顾一下课程C1_W2_LAB04中关于特征工程的讨论和实验)。原始特征是电影的发行年份以及以独热向量形式呈现的电影类型(共有14种类型)。工程特征是从用户评分得出的平均评分。注意：有多重类型的电影，其每一个genre都有一个train_vec.   

用户内容仅由工程特征组成，每个用户都会计算一个同genre平均评分。此外，用户ID、评分次数和平均评分，但它们不包含在训练或预测内容中，它们在解释数据时很有用。   

训练集由数据集中用户给出的所有评分组成。用户和电影向量作为一个训练集一起提供给上述网络。用户对其评分的所有电影使用相同的用户向量。    

下面，让我们加载并展示一些数据。

In [2]:
# 加载数据，设置配置变量
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()
print(f"item_train:{item_train.shape}")
print(f"user_train:{user_train.shape}")
print(f"y_train:{y_train.shape}")
print(f"item_vecs:{item_vecs.shape}")

num_user_features = user_train.shape[1] - 3  # remove userid, rating count and ave rating during training
num_item_features = item_train.shape[1] - 1  # remove movie id at train time
uvs = 3  # user genre vector start
ivs = 3  # item genre vector start
u_s = 3  # 训练中使用的列起始位置，用户
i_s = 1  # 训练中使用的列起始位置，电影
scaledata = True  # 如果为真，则对数据应用标准缩放器
print(f"Number of training vectors: {len(item_train)}")

item_train:(58187, 17)
user_train:(58187, 17)
y_train:(58187,)
item_vecs:(1883, 17)
Number of training vectors: 58187


对数据的解读：
$$
\begin{aligned}
用户数:&n_u=395 \\
电影数:&n_m=694 \\
第j个电影的类型数:&m_j \\
\text{item vector}行数:&1883=\sum\limits_{j=1}^{n_m}m_j\\
第i个用户给第j个电影了分:&r(i,j)=1 \\
\text{train}数据行数:&58187=\sum\limits_{j=1}^{n_m}\sum\limits_{(i,j):r(i,j)=1}m_j\\
\end{aligned}
$$

这里没搞明白，为什么有多重类型的电影要被拆成多行？

部分用户和电影特征在训练中未被使用。以下，方括号“[]”中的特征，如“用户ID”、“评分次数”和“平均评分”，在模型训练和使用时不包含在内。请注意，对于所有被评分的电影，用户向量是相同的。

In [3]:
pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

[user id],[rating count],[rating ave],Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Horror,Mystery,Romance,Sci-Fi,Thriller
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9


In [4]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

[movie id],year,ave rating,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Horror,Mystery,Romance,Sci-Fi,Thriller
6874,2003,4.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
6874,2003,4.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
6874,2003,4.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
8798,2004,3.8,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8798,2004,3.8,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [5]:
print(f"y_train[:5]: {y_train[:5]}")

y_train[:5]: [4.  4.  4.  3.5 3.5]


从上述内容可知，   
电影6874是一部2003年上映的动作片，   
用户2对动作片的平均评分是3.9分，    
电影6874还被归类为犯罪和惊悚类型，   
MovieLens的用户给这部电影的平均评分是4分。   
   
一个训练样本，由这两个表中的一行数据和y_train中的一个评分组成。

<a name="2.2"></a>
### 2.2 准备训练数据
回想一下，在C1_W2，你探索了特征缩放作为一种改善收敛性的方法。我们将使用[scikit learn标准缩放器](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)对输入特征进行缩放。这在C1_W2_LAB5中使用过。下面还展示了逆变换以生成原始输入。

In [6]:
# scale training data
if scaledata:
    item_train_save = item_train
    user_train_save = user_train

    scalerItem = StandardScaler()
    scalerItem.fit(item_train)
    item_train = scalerItem.transform(item_train)

    scalerUser = StandardScaler()
    scalerUser.fit(user_train)
    user_train = scalerUser.transform(user_train)

    print(np.allclose(item_train_save, scalerItem.inverse_transform(item_train)))
    print(np.allclose(user_train_save, scalerUser.inverse_transform(user_train)))

True
True


为了便于我们评估结果，我们将按照C2_W3所讨论的那样，把数据划分为训练集和测试集。这里我们将使用
[sklearn的train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
来划分和打乱数据。请注意，将初始随机状态设置为相同的值可确保对item、user和y进行相同的打乱。 

In [7]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test  data shape: {item_test.shape}")

movie/item training data shape: (46549, 17)
movie/item test  data shape: (11638, 17)


经过缩放和随机打乱的数据现在均值为零。

In [8]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

[user id],[rating count],[rating ave],Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Horror,Mystery,Romance,Sci-Fi,Thriller
1,0,0.6,0.7,0.6,0.6,0.7,0.7,0.5,0.7,0.2,0.3,0.3,0.5,0.5,0.8,0.5
0,0,1.6,1.5,1.7,0.9,1.0,1.4,0.8,-1.2,1.2,1.2,1.6,0.9,1.4,1.2,1.0
0,0,0.8,0.6,0.7,0.5,0.6,0.6,0.3,-1.2,0.7,0.8,0.9,0.6,0.2,0.6,0.6
1,0,-0.1,0.2,-0.1,0.3,0.7,0.3,0.2,1.0,-0.5,-0.7,-2.1,0.5,0.7,0.3,0.0
-1,0,-1.3,-0.8,-0.8,0.1,-0.1,-1.1,-0.9,-1.2,-1.5,-0.6,-0.5,-0.6,-0.9,-0.4,-0.9


使用最小 - 最大缩放器（Min Max Scaler）对目标评分进行缩放，使目标值介于 -1 和 1 之间。我们使用 scikit - learn 库，因为它具有逆变换功能。[scikit learn 最小 - 最大缩放器](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

In [9]:
scaler = MinMaxScaler((-1, 1))
scaler.fit(y_train.reshape(-1, 1))
ynorm_train = scaler.transform(y_train.reshape(-1, 1))
ynorm_test = scaler.transform(y_test.reshape(-1, 1))
print(ynorm_train.shape, ynorm_test.shape)

(46549, 1) (11638, 1)


<a name="3"></a>
## 3 - 基于内容过滤的神经网络
现在，让我们按照上图所述构建一个神经网络。它将有两个通过点积组合的网络。你将构建这两个网络。在这个例子中，它们将是相同的。请注意，这些网络不一定要相同。如果用户内容比电影内容大得多，你可能会选择相对于电影网络增加用户网络的复杂度。在这种情况下，内容相似，所以网络是相同的。
- 使用Keras顺序模型
    - 第一层是具有256个单元和relu激活函数的全连接层。
    - 第二层是具有128个单元和relu激活函数的全连接层。
    - 第三层是具有`num_outputs`个单元且激活函数为线性或无激活函数的全连接层。

网络的其余部分将由系统提供。提供的代码不使用Keras顺序模型，而是使用Keras [函数式API](https://keras.io/guides/functional_api/)。这种格式在组件的互连方式上提供了更大的灵活性。


In [11]:
# Public tests
from public_tests import *
test_tower(user_NN)
test_tower(item_NN)

All tests passed!
All tests passed!


In [10]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###   
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

# 创建user input并指向基础网络
input_user = tf.keras.layers.Input(shape=(num_user_features,))
vu = user_NN(input_user)
vu = tf.keras.layers.Lambda(
    lambda x: tf.linalg.l2_normalize(x, axis=1)
)(vu)

# 创建movie input并指向基础网络
input_item = tf.keras.layers.Input(shape=(num_item_features,))
vm = item_NN(input_item)
vm = tf.keras.layers.Lambda(
    lambda x: tf.linalg.l2_normalize(x, axis=1)
)(vm)

# 计算向量vu和vm的点积
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# 告诉Model()，输入是input_user和input_item这两个张量，输出是output
model = Model((input_user, input_item), output)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 14)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32)        │     40,864 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 32)        │     41,376 │ input_layer_2[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 32)        │          0 │ sequential[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 32)        │          0 │ sequential_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ lambda[0][0],     │
│                     │                   │            │ lambda_1[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 82,240 (321.25 KB)

 Trainable params: 82,240 (321.25 KB)

 Non-trainable params: 0 (0.00 B)

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
  You can create a dense layer with a relu activation as shown.
    
```python     
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])
```    
<details>
    <summary><font size="2" color="darkblue"><b> Click for solution</b></font></summary>
    
```python 
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])
```
</details>
</details>

    


我们将使用均方误差损失和Adam优化器。

In [12]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt, loss=cost_fn)

In [13]:
tf.random.set_seed(1)
model.fit((user_train[:, u_s:], item_train[:, i_s:]), ynorm_train, epochs=30)

Epoch 1/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1295  
Epoch 2/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1171
Epoch 3/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1149  
Epoch 4/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1135
Epoch 5/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1122
Epoch 6/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1112
Epoch 7/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1103
Epoch 8/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 1s 975us/step - loss: 0.1091
Epoch 9/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1080
Epoch 10/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1072
Epoch 11/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1065
Epoch 12/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1059
Epoch 13/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1052
Epoch 14/30
1455/1455 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1046  
Epoch 15/30
1455/1455

评估模型以确定测试数据上的损失。它与训练损失相当，表明该模型没有严重过拟合训练数据。

In [14]:
model.evaluate((user_test[:, u_s:], item_test[:, i_s:]), ynorm_test)

364/364 ━━━━━━━━━━━━━━━━━━━━ 0s 471us/step - loss: 0.1058


0.10563339293003082

<a name="3.1"></a>
### 3.1 预测
接下来，你将使用模型在多种情况下进行预测。 
#### 针对新用户的预测
首先，我们将创建一个新用户，并让模型为该用户推荐电影。在示例用户内容上尝试此示例后，你可以随意更改用户内容以匹配自己的偏好，看看模型会给出什么推荐。请注意，评分介于0.5到5.0之间（包括0.5和5.0），以0.5为步长递增。 

In [15]:
new_user_id = 5000
new_rating_ave = 1.0
new_action = 1.0
new_adventure = 1
new_animation = 1
new_childrens = 1
new_comedy = 5
new_crime = 1
new_documentary = 1
new_drama = 1
new_fantasy = 1
new_horror = 1
new_mystery = 1
new_romance = 5
new_scifi = 5
new_thriller = 1
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

让我们看看新用户评分最高的电影。回想一下，用户向量中的类型偏好喜剧和爱情片。
下面，我们将使用一组电影向量“item_vecs”，其中为训练/测试集中的每部电影都提供了一个向量。这与上面的用户向量相匹配，并且缩放后的向量用于预测上述新用户对所有电影的评分。

In [16]:
# 生成并复制用户向量，以匹配数据集中电影的数量。
user_vecs = gen_user_vecs(user_vec,len(item_vecs))
print(user_vecs.shape)

(1883, 17)


In [17]:


# 缩放向量并对所有电影进行预测。返回按评分排序的结果。
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs,  item_vecs, model, u_s, i_s, 
                                                                       scaler, scalerUser, scalerItem, scaledata=scaledata)

print_pred_movies(sorted_ypu, sorted_user, sorted_items, movie_dict, maxcount = 10)

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


y_p,movie id,rating ave,title,genres
4.79858,7149,3.64706,Something's Gotta Give (2003),Comedy|Drama|Romance
4.79752,5673,3.62121,Punch-Drunk Love (2002),Comedy|Drama|Romance
4.79501,5066,3.5,"Walk to Remember, A (2002)",Drama|Romance
4.79276,8983,3.52,House of Flying Daggers (Shi mian mai fu) (2004),Action|Drama|Romance
4.79106,7137,3.55,"Cooler, The (2003)",Comedy|Drama|Romance
4.78774,8533,3.56579,"Notebook, The (2004)",Drama|Romance
4.7875,5992,3.7,"Hours, The (2002)",Drama|Romance
4.78722,8638,3.7,Before Sunset (2004),Drama|Romance
4.78605,8784,3.70833,Garden State (2004),Comedy|Drama|Romance
4.78603,7293,3.55319,50 First Dates (2004),Comedy|Romance


如果你确实在上面创建了一个用户，需要注意的是，**该网络的训练是为了预测A用户的评分，依据的是A用户的特征，而A用户的特征来自于A用户已经打的分。如果A用户只是为单个genre提供最高评分，而对其余类型提供最低评分，同时没有其他用户有类似的rating set，这对该网络来说可能没有意义。 

#### 对现有用户的预测。
我们来看看数据集中“用户36”的预测结果。我们可以将预测评分与模型的评分进行比较。请注意，具有多种类型的电影在训练数据中会多次出现。例如，《时光机器》有三种类型：冒险、动作、科幻

In [18]:
uid =  36 
# form a set of user vectors. This is the same vector, transformed and repeated.
user_vecs, y_vecs = get_user_vecs(uid, scalerUser.inverse_transform(user_train), item_vecs, user_to_genre)

# scale the vectors and make predictions for all movies. Return results sorted by rating.
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs, item_vecs, model, u_s, i_s, scaler, 
                                                                      scalerUser, scalerItem, scaledata=scaledata)
sorted_y = y_vecs[sorted_index]

#print sorted predictions
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, item_features, ivs, uvs, movie_dict, maxcount = 10)

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 670us/step


y_p,y,user,user genre ave,movie rating ave,title,genres
3.0,3.0,36,3.00,2.86,"Time Machine, The (2002)",Adventure
2.9,3.0,36,3.00,2.86,"Time Machine, The (2002)",Action
2.8,3.0,36,3.00,2.86,"Time Machine, The (2002)",Sci-Fi
1.9,1.5,36,1.75,3.52,Road to Perdition (2002),Crime
1.9,2.0,36,1.75,3.52,Gangs of New York (2002),Crime
1.9,1.0,36,1.50,4.00,"Beautiful Mind, A (2001)",Drama
1.8,1.0,36,1.00,4.00,"Beautiful Mind, A (2001)",Romance
1.5,1.5,36,1.50,3.52,Road to Perdition (2002),Drama
1.5,2.0,36,1.50,3.52,Gangs of New York (2002),Drama


#### 寻找相似项目
上述神经网络生成两个特征向量，一个用户特征向量$v_u$和一个电影特征向量$v_m$。这些是32维向量，其值难以解读。然而，相似的项目会有相似的向量。这些信息可用于进行推荐。例如，如果一个用户对《玩具总动员3》评价很高，就可以通过选择具有相似电影特征向量的电影来推荐类似电影。
一种相似性度量是两个向量$\mathbf{v_m^{(k)}}$和$\mathbf{v_m^{(i)}}$之间的平方距离:
$$\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}$$

<a name="ex01"></a>
### 练习1
编写一个函数来计算平方距离。

In [19]:
# GRADED_FUNCTION: sq_dist
# UNQ_C2
def sq_dist(a,b):
    """
    Returns the squared distance between two vectors
    Args:
      a (ndarray (n,)): vector with n features
      b (ndarray (n,)): vector with n features
    Returns:
      d (float) : distance
    """
    ### START CODE HERE ###
    diff = a - b
    squared_diff = np.square(diff)
    d = np.sum(squared_diff)
    ### END CODE HERE ###     
    return (d)

In [20]:
# Public tests
test_sq_dist(sq_dist)

All tests passed!


In [21]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1)}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2)}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3)}")

squared distance between a1 and b1: 0.0
squared distance between a2 and b2: 0.030000000000000054
squared distance between a3 and b3: 2


<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
  While a summation is often an indication a for loop should be used, here the subtraction can be element-wise in one statement. Further, you can utilized np.square to square, element-wise, the result of the subtraction. np.sum can be used to sum the squared elements.
    
</details>

    


在模型训练时，可以计算一次电影之间的距离矩阵，然后在进行新推荐时重复使用，无需重新训练。模型训练完成后的第一步，是为每部电影获取电影特征向量$v_m$ 。为此，我们将使用训练好的`item_NN`并构建一个小型模型，以便我们将电影向量输入其中，生成$v_m$ 。

In [22]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features,))    # input layer
vm_m = item_NN(input_item_m)                                       # use the trained item_NN
vm_m = tf.keras.layers.Lambda(                                   # 按照原始模型中的做法加入归一化处理
    lambda x: tf.linalg.l2_normalize(x, axis=1)
)(vm_m)                        
model_m = Model(input_item_m, vm_m)                                
model_m.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 32)             │        41,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_2 (Lambda)               │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,376 (161.62 KB)

 Trainable params: 41,376 (161.62 KB)

 Non-trainable params: 0 (0.00 B)

一旦你有了一个电影模型，就可以通过使用该模型，以一组物品/电影向量作为输入进行预测，从而创建一组电影特征向量。
`item_vecs`是所有电影向量的集合。请记住，同一部电影会因其每个类型而以单独的向量形式出现。必须对其进行缩放才能与训练好的模型配合使用。预测结果是每部电影的一个32维特征向量。

In [23]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 746us/step
size of all predicted movie feature vectors: (1883, 32)


现在，让我们计算一个矩阵，该矩阵表示每部电影特征向量与所有其他电影特征向量之间的平方距离：
<figure>
    <left> <img src="./images/distmatrix.PNG"   style="width:400px;height:225px;" ></center>
</figure>

然后，我们可以通过找出每行中的最小值来找到最相似的电影。我们将使用[NumPy掩码数组](https://numpy.org/doc/1.21/user/tutorial-ma.html)来避免选择同一部电影。对角线上的掩码值将不包含在计算中。

In [24]:
count = 50
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # mask the diagonal

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    genre1,_  = get_item_genre(item_vecs[i,:], ivs, item_features)
    genre2,_  = get_item_genre(item_vecs[min_idx,:], ivs, item_features)

    disp.append( [movie_dict[movie1_id]['title'], genre1,
                  movie_dict[movie2_id]['title'], genre2]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow", floatfmt=[".1f", ".1f", ".0f", ".2f", ".2f"])
table

movie1,genres,movie2,genres
Save the Last Dance (2001),Drama,John Q (2002),Drama
Save the Last Dance (2001),Romance,Saving Silverman (Evil Woman) (2001),Romance
"Wedding Planner, The (2001)",Comedy,National Lampoon's Van Wilder (2002),Comedy
"Wedding Planner, The (2001)",Romance,Mr. Deeds (2002),Romance
Hannibal (2001),Horror,Final Destination 2 (2003),Horror
Hannibal (2001),Thriller,"Sum of All Fears, The (2002)",Thriller
Saving Silverman (Evil Woman) (2001),Comedy,Cats & Dogs (2001),Comedy
Saving Silverman (Evil Woman) (2001),Romance,Save the Last Dance (2001),Romance
Down to Earth (2001),Comedy,Joe Dirt (2001),Comedy
Down to Earth (2001),Fantasy,"Haunted Mansion, The (2003)",Fantasy


结果表明，该模型将推荐同一类型的电影。

<a name="4"></a>
## 4 - 恭喜！<img align="left" src="./images/film_award.png" style=" width:40px;">
你已经完成了一个基于内容的推荐系统。    
这种结构是许多商业推荐系统的基础。如果有可用的用户信息，用户内容可以大幅扩展，纳入更多关于用户的信息。物品并不局限于电影。这可以用于推荐任何物品，如书籍、汽车，或者与你“购物车”中的物品相似的物品。